# Data Cleaning for University Campus Health Surveillance Project
# Author: Ari M.
# Description: Load the simulated EMR datasets and perform basic validation before using them for analysis.
# Datasets: patients.csv, tests.csv, vaccinations.csv

## Approach

Before starting analysis, I want to validate that the datasets are structured correctly and do not contain obvious data quality issues.

Since this data was simulated in Excel, I expect:
- Date fields may need conversion
- There could be minor inconsistencies
- Relationships between tables should be verified

I will go step-by-step to confirm the data is reliable before using it for analysis.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## Define File Paths

The datasets were generated earlier in the project to simulate an electronic medical record (EMR) system used by a university health clinic.

These datasets include:

• **patients.csv** – demographic information  
• **tests.csv** – diagnostic testing records  
• **vaccinations.csv** – vaccination records

All datasets are stored in the **data/processed/** folder.

In [2]:
patients_path = "../data/processed/patients.csv"
tests_path = "../data/processed/tests.csv"
vaccinations_path = "../data/processed/vaccinations.csv"

In [3]:
patients = pd.read_csv(patients_path)
tests = pd.read_csv(tests_path)
vaccinations = pd.read_csv(vaccinations_path)

print("Datasets loaded successfully.")

Datasets loaded successfully.


At this point, I just want to confirm that the datasets loaded correctly and contain a reasonable number of records before continuing.

In [4]:
# Quick check: confirm datasets are not empty
print(len(patients), len(tests), len(vaccinations))

7088 25103 10075


In [5]:
print("Patients dataset shape:", patients.shape)
print("Tests dataset shape:", tests.shape)
print("Vaccinations dataset shape:", vaccinations.shape)

Patients dataset shape: (7088, 7)
Tests dataset shape: (25103, 7)
Vaccinations dataset shape: (10075, 6)


## Preview Dataset Structure 
Want to confirm that columns appear as expected and values look reasonable.

In [6]:
patients.head()

,patient_id,age,gender,staff_or_student_status,residence_type,pt_has_test_y_n,pt_has_vaccination_y_n
0,1,25,Female,Student,On-campus,Yes,No
1,2,25,Male,Student,Off-campus,Yes,No
2,3,23,Male,Student,On-campus,Yes,Yes
3,4,20,Male,Student,On-campus,Yes,Yes
4,5,25,Female,Student,Off-campus,Yes,No


In [7]:
tests.head()

,test_id,patient_id,disease,test_date,test_result,report_date,lab_source
0,1,4046,Norovirus,43664,Negative,43666,Campus Lab
1,2,6291,Gonorrhea,44755,Negative,44757,Hospital Lab
2,3,6255,COVID-19,43855,Negative,43857,Campus Lab
3,4,1613,Chlamydia,43704,Negative,43706,Hospital Lab
4,5,7005,COVID-19,43974,Negative,43976,Hospital Lab


In [8]:
vaccinations.head()

,vaccination_id,patient_id,vaccine_type,dose_number,vaccination_date,vaccination_year
0,1,3706,Influenza,1,45506,2024
1,2,4788,Influenza,1,44310,2021
2,3,2566,COVID-19,2,44608,2022
3,4,976,COVID-19,1,45455,2024
4,5,4925,COVID-19,3,44228,2021


### Observation

At a quick glance, the datasets appear structured as expected.

Key identifiers such as patient_id, test_id, and vaccination_id are present, which should allow joins across tables.

However, I have not yet verified:
- whether all IDs match across tables
- whether dates are valid
- whether there are hidden inconsistencies

I will check these in the next steps.

## Inspect Column Data Types

Checking column data types to see whether dates are being stored as numbers or strings.

In [9]:
patients.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7088 entries, 0 to 7087
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   patient_id               7088 non-null   int64 
 1   age                      7088 non-null   int64 
 2   gender                   7088 non-null   object
 3   staff_or_student_status  7088 non-null   object
 4   residence_type           7088 non-null   object
 5   pt_has_test_y_n          7088 non-null   object
 6   pt_has_vaccination_y_n   7088 non-null   object
dtypes: int64(2), object(5)
memory usage: 387.8+ KB


In [10]:
tests.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25103 entries, 0 to 25102
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   test_id      25103 non-null  int64 
 1   patient_id   25103 non-null  int64 
 2   disease      25103 non-null  object
 3   test_date    25103 non-null  int64 
 4   test_result  25103 non-null  object
 5   report_date  25103 non-null  int64 
 6   lab_source   25103 non-null  object
dtypes: int64(4), object(3)
memory usage: 1.3+ MB


In [11]:
vaccinations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10075 entries, 0 to 10074
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   vaccination_id    10075 non-null  int64 
 1   patient_id        10075 non-null  int64 
 2   vaccine_type      10075 non-null  object
 3   dose_number       10075 non-null  int64 
 4   vaccination_date  10075 non-null  int64 
 5   vaccination_year  10075 non-null  int64 
dtypes: int64(5), object(1)
memory usage: 472.4+ KB


### Observation

Most columns appear correctly typed.

However, date fields are stored as numeric values, likely due to Excel’s serial date format.

If not converted properly, this would break any time-based analysis (e.g., trends, seasonality).

I will convert these fields and then verify the results.

## Convert Excel Serial Dates

The datasets were originally generated in Excel, which stores dates as numbers.

I will convert these values into proper datetime format so they can be used for time-based analysis.

In [12]:
tests["test_date"] = pd.to_datetime(
    tests["test_date"], origin="1899-12-30", unit="D"
)

tests["report_date"] = pd.to_datetime(
    tests["report_date"], origin="1899-12-30", unit="D"
)

vaccinations["vaccination_date"] = pd.to_datetime(
    vaccinations["vaccination_date"], origin="1899-12-30", unit="D"
)

In [13]:
tests[["test_date", "report_date"]].head()

,test_date,report_date
0,2019-07-18,2019-07-20
1,2022-07-13,2022-07-15
2,2020-01-25,2020-01-27
3,2019-08-27,2019-08-29
4,2020-05-23,2020-05-25


In [14]:
vaccinations[["vaccination_date"]].head()

,vaccination_date
0,2024-08-02
1,2021-04-24
2,2022-02-16
3,2024-06-12
4,2021-02-01


In [15]:
# I want to make sure dates fall within reasonable bounds for the study period
print("Test dates range:", tests["test_date"].min(), "to", tests["test_date"].max())
print("Report dates range:", tests["report_date"].min(), "to", tests["report_date"].max())
print("Vaccination dates range:", vaccinations["vaccination_date"].min(), "to", vaccinations["vaccination_date"].max())

Test dates range: 2019-01-01 00:00:00 to 2024-12-28 00:00:00
Report dates range: 2019-01-02 00:00:00 to 2024-12-30 00:00:00
Vaccination dates range: 2020-01-01 00:00:00 to 2024-12-28 00:00:00


In [16]:
# I want to make sure the report_date is always after test_date
invalid_dates = tests[tests["report_date"] < tests["test_date"]]
print("Invalid date records:", invalid_dates.shape)

Invalid date records: (0, 7)


### Observation

Date columns now appear in standard datetime format.

The validation check confirms that report_date occurs after test_date, which is expected in a real-world workflow.

This gives me reasonable confidence that the date conversion worked as expected, although I may revisit this later if anything unusual shows up during analysis.

## Check for Missing Values

I will check each dataset to confirm that required fields are populated so that the analysis is accurate.

In [17]:
print("Missing values in patients:")
print(patients.isna().sum())

print("\nMissing values in tests:")
print(tests.isna().sum())

print("\nMissing values in vaccinations:")
print(vaccinations.isna().sum())

Missing values in patients:
patient_id                 0
age                        0
gender                     0
staff_or_student_status    0
residence_type             0
pt_has_test_y_n            0
pt_has_vaccination_y_n     0
dtype: int64

Missing values in tests:
test_id        0
patient_id     0
disease        0
test_date      0
test_result    0
report_date    0
lab_source     0
dtype: int64

Missing values in vaccinations:
vaccination_id      0
patient_id          0
vaccine_type        0
dose_number         0
vaccination_date    0
vaccination_year    0
dtype: int64


### Observation

No missing values were found across the datasets.

This suggests the data generation step worked as expected.

Even though no missing values were found, I want to keep this step because in real healthcare datasets missing data is very common and can significantly impact analysis.

## Validate Age Ranges

The dataset includes two population groups:

• **Students** (expected age range: 18–25)  
• **Staff** (expected age range: 26–65)

I will check for any records that fall outside these ranges.

In [18]:
invalid_ages = patients[
    ((patients["staff_or_student_status"] == "Student") &
     ((patients["age"] < 18) | (patients["age"] > 25))) |
    ((patients["staff_or_student_status"] == "Staff") &
     ((patients["age"] < 26) | (patients["age"] > 65)))
]

invalid_ages

,patient_id,age,gender,staff_or_student_status,residence_type,pt_has_test_y_n,pt_has_vaccination_y_n


### Observation

No invalid age records were detected.

Students are within the expected 18–25 age range, while staff is within the 26–65 range.

This validation confirms that the simulation logic worked correctly.

In real-world data, I would expect some outliers or incorrect entries that would require correction or further investigation.

## Validate Relationships Between Tables

Since this simulates an EMR system, I want to confirm that all records in tests and vaccinations link to valid patients.

In [19]:
# Tests to Patients
tests_missing_patients = tests[~tests["patient_id"].isin(patients["patient_id"])]
print("Tests with missing patients:", tests_missing_patients.shape)

# Vaccinations to Patients
vacc_missing_patients = vaccinations[~vaccinations["patient_id"].isin(patients["patient_id"])]
print("Vaccinations with missing patients:", vacc_missing_patients.shape)

Tests with missing patients: (0, 7)
Vaccinations with missing patients: (2, 6)


In [20]:
vacc_missing_patients.head()

,vaccination_id,patient_id,vaccine_type,dose_number,vaccination_date,vaccination_year
2890,2891,7089,COVID-19,2,2024-07-19,2024
9969,9970,7089,COVID-19,3,2024-02-16,2024


### Observation

Two vaccination records reference patient_id = 7089, which does not exist in the patients table. I didn’t expect to find issues here since the data is simulated, so this was a bit surprising.

This issue likely occurred during data generation, where the random assignment allowed a patient_id outside the valid range.

This type of mismatch is common in real-world healthcare systems when:
- data is generated or entered incorrectly
- systems are not fully synchronized
- validation constraints are missing

Since these records cannot be linked to a valid patient, I’ll remove them so they don’t cause issues later during analysis.

In [21]:
# Remove invalid vaccination records
vaccinations = vaccinations[
    vaccinations["patient_id"].isin(patients["patient_id"])
]

print("Vaccinations shape after cleaning:", vaccinations.shape)

Vaccinations shape after cleaning: (10073, 6)


In [22]:
# Investigate invalid patient IDs
vacc_missing_patients["patient_id"].value_counts()

patient_id
7089    2
Name: count, dtype: int64

In a production system, this issue would ideally be prevented by enforcing referential integrity constraints at the database level.

### Note
I identified and removed 2 vaccination records with invalid patient_id values

## Duplicate Check

In [23]:
print("Duplicate patients:", patients.duplicated().sum())
print("Duplicate tests:", tests.duplicated().sum())
print("Duplicate vaccinations:", vaccinations.duplicated().sum())

Duplicate patients: 0
Duplicate tests: 0
Duplicate vaccinations: 0


### Observation

No duplicate records were found.

This is expected given the simulation, but in real-world datasets duplicates are common and I would handle them carefully.

## Quick Data Distribution Checks

Before moving to analysis, I will perform several quick checks to understand how data is distributed across key categories.

In [24]:
tests["disease"].value_counts()

disease
Gonorrhea    5246
Influenza    5222
Norovirus    5184
Chlamydia    5179
COVID-19     4272
Name: count, dtype: int64

In [25]:
patients["staff_or_student_status"].value_counts()

staff_or_student_status
Student    5316
Staff      1772
Name: count, dtype: int64

In [26]:
vaccinations["vaccine_type"].value_counts()

vaccine_type
Influenza    5125
COVID-19     4948
Name: count, dtype: int64

### Observation

Disease counts are relatively balanced across categories, which aligns with how the dataset was simulated. I’m not completely sure if this distribution is realistic, but it looks reasonable for a simulated dataset.

In real-world data, I would probably expect more variation (for example, fewer STI tests compared to respiratory illnesses), so this is something I’ll keep in mind during analysis.

The student population makes up the majority of records, which reflects a typical university setting.

In [27]:
patients["age"].describe()

count    7088.000000
mean       27.497319
std        11.970096
min        18.000000
25%        20.000000
50%        23.000000
75%        25.250000
max        65.000000
Name: age, dtype: float64

### Observation

The age distribution reflects two distinct groups (students and staff), which is consistent with how the dataset was designed.

This segmentation will be useful later when analyzing disease risk across different populations.

## Save Cleaned Datasets

After validating the data, I'm going to export cleaned versions of the datasets to the **data/cleaned/** folder.

These cleaned files will be used for the next stage of the project: exploratory analysis and visualization.

In [28]:
patients_cleaned_path = "../data/cleaned/patients_cleaned.csv"
tests_cleaned_path = "../data/cleaned/tests_cleaned.csv"
vaccinations_cleaned_path = "../data/cleaned/vaccinations_cleaned.csv"

patients.to_csv(patients_cleaned_path, index=False)
tests.to_csv(tests_cleaned_path, index=False)
vaccinations.to_csv(vaccinations_cleaned_path, index=False)

print("Cleaned datasets saved successfully.")

Cleaned datasets saved successfully.


## Data Cleaning Summary

During this step, I validated and cleaned the datasets to ensure they are reliable for analysis.

Key actions:

- Converted Excel serial date fields into proper datetime format
- Verified that report dates occur after test dates
- Checked for missing values (none found)
- Validated age ranges for students and staff
- Identified and removed 2 vaccination records with invalid patient_id values
- Confirmed no duplicate records
- Performed basic distribution checks to understand the data

One issue I encountered was a mismatch between vaccination and patient records, which I resolved by removing invalid entries.

The datasets now appear consistent and ready for exploratory analysis, although I may revisit certain assumptions if issues arise later.